In [77]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import time

In [78]:
header = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'}
webpage = requests.get('https://www.ambitionbox.com/list-of-companies?page=1',headers = header).text
soup = BeautifulSoup(webpage,'lxml')

In [30]:
h2list = soup.find_all('h2' , class_ = 'companyCardWrapper__companyName')
for i in h2list:
    print(i.text.strip())

In [29]:
for i in soup.find_all('div' , class_="rating_text"):
     print(i.text.strip())
    

20

In [79]:
company = soup.find_all('div' , class_ = 'companyCardWrapper')
name = []
ratings = []
reviews = []
companytype=[]
locations = []
Highly_Rated_For = []
Critically_Rated_For = []
for i in company:
    name.append(i.find('h2').text.strip())
    ratings.append(i.find('div' , class_="rating_text").text.strip())
    reviews.append(i.find('span' , class_="companyCardWrapper__ActionCount").text.strip())
    companytype.append(i.find('span' , class_="companyCardWrapper__interLinking").text.strip().split('|')[0].strip())
    locations.append(i.find('span' , class_="companyCardWrapper__interLinking").text.strip().split('|')[1].strip())
    Highly_Rated_For.append(i.find_all('span' , class_="companyCardWrapper__ratingValues")[0].text.strip())
    try:
        Critically_Rated_For.append(i.find_all('span' , class_="companyCardWrapper__ratingValues")[1].text.strip())
    except:
        Critically_Rated_For.append(np.nan)

        

In [80]:
d = {'name' : name , 'ratings' : ratings , 'reviews' : reviews , 'companytype' : companytype , 'locations' : locations ,'Highly_Rated_For' : Highly_Rated_For , 'Critically_Rated_For' : Critically_Rated_For  }
df = pd.DataFrame(d)

In [69]:
df

,name,ratings,reviews,companytype,locations,Highly_Rated_For,Critically_Rated_For
0,TCS,3.3,1.1L,IT Services & Consulting,Bengaluru +440 other locations,Job Security,"Promotions / Appraisal, Salary & Benefits, Wor..."
1,Accenture,3.7,72.5k,IT Services & Consulting,Bengaluru +257 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
2,Wipro,3.6,64.4k,IT Services & Consulting,Hyderabad +371 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
3,Cognizant,3.7,60.7k,IT Services & Consulting,Hyderabad +233 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
4,Capgemini,3.7,52.4k,IT Services & Consulting,Bengaluru +182 other locations,"Work Life Balance, Job Security","Promotions / Appraisal, Salary & Benefits, Wor..."
5,HDFC Bank,3.8,51.5k,Banking,Mumbai +1839 other locations,"Job Security, Skill Development / Learning","Promotions / Appraisal, Salary & Benefits"
6,Infosys,3.5,48.2k,IT Services & Consulting,Bengaluru +248 other locations,Job Security,"Promotions / Appraisal, Salary & Benefits, Wor..."
7,ICICI Bank,3.9,45.5k,Banking,Mumbai +1441 other locations,"Job Security, Promotions / Appraisal, Skill De...",NaN
8,HCLTech,3.4,45.4k,IT Services & Consulting,Chennai +232 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
9,Tech Mahindra,3.4,42.9k,IT Services & Consulting,Hyderabad +331 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN


In [84]:
final = pd.DataFrame()
all_pages_data = []

for j in range(1,111):
    url = 'https://www.ambitionbox.com/list-of-companies?page={}'.format(j)
    header = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'}
    webpage = requests.get(url,headers = header).text

    soup = BeautifulSoup(webpage,'lxml')
    company = soup.find_all('div' , class_ = 'companyCardWrapper')
    name = []
    ratings = []
    reviews = []
    companytype=[]
    locations = []
    Highly_Rated_For = []
    Critically_Rated_For = []
    
    for i in company:
        # 1. Name
        try:
            name.append(i.find('h2').text.strip())
        except:
            name.append(np.nan)
    
        # 2. Ratings (Fixes your current error)
        try:
            ratings.append(i.find('div', class_="rating_text").text.strip())
        except:
            ratings.append(np.nan)
    
        # 3. Reviews
        try:
            reviews.append(i.find('span', class_="companyCardWrapper__ActionCount").text.strip())
        except:
            reviews.append(np.nan)
            
        # 4. Company Type & Locations (The split fix)
        try:
            interlinking = i.find('span', class_="companyCardWrapper__interLinking").text.strip().split('|')
            companytype.append(interlinking[0].strip() if len(interlinking) > 0 else np.nan)
            locations.append(interlinking[1].strip() if len(interlinking) > 1 else np.nan)
        except:
            companytype.append(np.nan)
            locations.append(np.nan)

        # 5. Highly Rated For
        try:
            rating_tags = i.find_all('span', class_="companyCardWrapper__ratingValues")
            Highly_Rated_For.append(rating_tags[0].text.strip() if len(rating_tags) > 0 else np.nan)
        except:
            Highly_Rated_For.append(np.nan)
        
        # 6. Critically Rated For    
        try:
            Critically_Rated_For.append(rating_tags[1].text.strip() if len(rating_tags) > 1 else np.nan)
        except:
            Critically_Rated_For.append(np.nan)
            
    d = {'name' : name , 'ratings' : ratings , 'reviews' : reviews , 'companytype' : companytype , 'locations' : locations ,'Highly_Rated_For' : Highly_Rated_For , 'Critically_Rated_For' : Critically_Rated_For  }
    df = pd.DataFrame(d)
    all_pages_data.append(df)
    
final = pd.concat(all_pages_data , ignore_index=True)     
    

In [85]:
final

,name,ratings,reviews,companytype,locations,Highly_Rated_For,Critically_Rated_For
0,TCS,3.3,1.1L,IT Services & Consulting,Bengaluru +440 other locations,Job Security,"Promotions / Appraisal, Salary & Benefits, Wor..."
1,Accenture,3.7,72.5k,IT Services & Consulting,Bengaluru +257 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
2,Wipro,3.6,64.4k,IT Services & Consulting,Hyderabad +371 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
3,Cognizant,3.7,60.7k,IT Services & Consulting,Hyderabad +233 other locations,"Promotions / Appraisal, Salary & Benefits, Wor...",NaN
4,Capgemini,3.7,52.4k,IT Services & Consulting,Bengaluru +182 other locations,"Work Life Balance, Job Security","Promotions / Appraisal, Salary & Benefits, Wor..."
...,...,...,...,...,...,...,...
2195,Kirby Building Systems,4.0,488,Building Material,Hyderabad +38 other locations,"Job Security, Work Life Balance, Company Culture",Promotions / Appraisal
2196,Trimax IT Infrastructure Services,3.6,487,IT Services & Consulting,Mumbai +70 other locations,"Promotions / Appraisal, Salary & Benefits, Com...",NaN
2197,Juniper Networks,4.2,487,Hardware & Networking,Bengaluru +8 other locations,"Company Culture, Work Life Balance, Salary & B...",NaN
2198,Maersk Line,4.0,487,Logistics,Pune +27 other locations,"Work Life Balance, Company Culture, Work Satis...",Promotions / Appraisal


In [86]:
final.to_csv('ambitionbox_companies.csv', index=False)